Importing Dependencies

In [1]:
import pandas as pd
import numpy as np

*   Import Dataset -> Dialogues
*   The type of character a person has depends on the dialogues they spoke
*   Read the dialogues and determine the traits of character






In [31]:
# dataframe with pandas
df = pd.read_json(r'data\script-bag-of-words.json')

In [4]:
df.tail()

,episodeAlt,seasonNum,episodeNum,episodeTitle,text
68,S8E2,8,2,A Knight of the Seven Kingdoms,"[{'name': 'Daenerys Targaryen', 'text': 'About..."
69,S8E3,8,3,The Long Night,"[{'name': 'Northman #1', 'text': 'Oi!'}, {'nam..."
70,S8E4,8,4,The Last of the Starks,"[{'name': 'Jon Snow', 'text': 'And Everyone It..."
71,S8E5,8,5,The Bells,"[{'name': 'Lord Varys', 'text': 'And? Come Not..."
72,S8E6,8,6,The Iron Throne,"[{'name': 'Tyrion Lannister', 'text': 'I'll fi..."


In [32]:
df.iloc[-1]['text']

# data is in this format -> dictinary in a list

[{'name': 'Tyrion Lannister', 'text': "I'll find later. you"},
 {'name': 'Jon Snow', 'text': "It's Let me men not safe. send some with you."},
 {'name': 'Tyrion Lannister', 'text': "I'm alone. going"},
 {'name': 'Grey Worm',
  'text': 'Daenerys I In Queen, Targaryen, die. name of one sentence the the to true you'},
 {'name': 'Jon Snow',
  'text': "Grey It's These Worm! are men over. prisoners."},
 {'name': 'Grey Worm',
  'text': "It Queen's are defeated. enemies is not over the until"},
 {'name': 'Davos Seaworth',
  'text': "How They're be? defeated do knees. more much on their them to want you"},
 {'name': 'Grey Worm', 'text': 'They are breathing.'},
 {'name': 'Davos Seaworth', 'text': 'Look We around friend. won. you,'},
 {'name': 'Grey Worm', 'text': "I commands, my not obey queen's yours."},
 {'name': 'Jon Snow', 'text': "And Queen's are commands? the what"},
 {'name': 'Grey Worm',
  'text': '"Kill Cersei Lannister." These They all are chose fight follow for free her. men. to who'}

Change the dataset pattern ... make it more clear

*   Add Charcter name
*   Add words(dialouge)
*   Add number of words





In [33]:
# df.iterrows() is a built-in function in Pandas
# It allows you to iterate over DataFrame rows as (index, Series) pairs.

dialouge = {}
for i, row in df.iterrows():
  for item in row['text']:
    if item['name'] in dialouge:
      # append
      dialouge[item['name']] += item['text']
    else:
      # create character
      dialouge[item['name']] = item['text']  + " "


In [34]:
len(dialouge) # gives total no of charcaters in GOT

817

Store key(names) and values(dialouges) in new DataFrame

In [35]:
new_df = pd.DataFrame()
new_df['character'] = dialouge.keys()
new_df['words'] = dialouge.values()

In [36]:
new_df.iloc[:,0:3].head()

,character,words
0,Will,"Easy, boy. I've I've Wildlings a a do ever in ..."
1,Waymar Royce,One They're What a and another before d'you ea...
2,Gared,Wall. We back head should the to Our They We W...
3,Jon Snow,Father's Go on. watching. And mother. yourBran...
4,Septa Mordane,"Fine Well always. as done. work, I Quite beaut..."


In [37]:
# count no of words each character spoke and add as a coloumn in new_df

new_df['num_words'] = new_df['words'].apply(lambda x:len(x.split()))
new_df = new_df.sort_values('num_words',ascending=False)
new_df = new_df.head(100)

# taking only top 100 characters out of 817 for low size & high effiecny


In [38]:
new_df.shape

(100, 3)

In [39]:
new_df.head()

,character,words,num_words
17,Tyrion Lannister,It Mmh. Northern about girls. is say the they ...,25924
13,Cersei Lannister,And And Casterly One Rock. When about afraid. ...,14294
3,Jon Snow,Father's Go on. watching. And mother. yourBran...,11488
20,Daenerys Targaryen,We've a and anything. asked been for for guest...,11202
12,Jaime Lannister,"As I It's brother, duty feel it's much. my sho...",10823


VECTORIZATION (Bag Of Word)

In [40]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(stop_words='english')
# stop_words is used to remove unwanted words(is,and,this,the) in statements

Embedding and Tokenization

In [41]:
embedding = cv.fit_transform(new_df['words']).toarray()

In [42]:
embedding.shape    # 100 characters and 15335 unique words on total

(100, 15335)

In [43]:
embedding = embedding.astype('float64')

Import t-SNE for visualization and dim reduction
TSNE is use for :
1.   understand patterns & relationships
2.   helps in clustering



In [44]:
from sklearn.manifold import TSNE

In [45]:
tsne = TSNE(n_components=2, verbose=1, random_state=123)
z = tsne.fit_transform(embedding)     # reduce embedding into 2D

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 100 samples in 0.005s...
[t-SNE] Computed neighbors for 100 samples in 0.128s...
[t-SNE] Computed conditional probabilities for sample 100 / 100
[t-SNE] Mean sigma: 14.377887
[t-SNE] KL divergence after 250 iterations with early exaggeration: 64.401108
[t-SNE] KL divergence after 800 iterations: 0.301372


In [46]:
z.shape

(100, 2)

In [47]:
# add new cols x and y
new_df['X'] = z.T[0]
new_df['Y'] = z.T[1]

In [48]:
new_df

,character,words,num_words,X,Y
17,Tyrion Lannister,It Mmh. Northern about girls. is say the they ...,25924,-3.342438,0.753784
13,Cersei Lannister,And And Casterly One Rock. When about afraid. ...,14294,-3.092457,0.733753
3,Jon Snow,Father's Go on. watching. And mother. yourBran...,11488,-2.974039,0.970400
20,Daenerys Targaryen,We've a and anything. asked been for for guest...,11202,-2.586169,0.337781
12,Jaime Lannister,"As I It's brother, duty feel it's much. my sho...",10823,-2.908238,0.824755
...,...,...,...,...,...
132,Rickard Karstark,Castle I'll King North! Red The They and can c...,466,2.763872,3.781367
57,Syrio Forel,"Tomorrow You are at be boy. here late, midday....",462,3.403694,2.542524
112,Kevan Lannister,Tyrion. Catelyn Golden Jaime Lords River River...,460,3.141036,2.772262
365,Mace Tyrell,"From Grace, House Margaery May Reach, Tyrell Y...",450,3.128660,2.601798


Plot the Graph

In [26]:
import plotly.express as px
fig = px.scatter(new_df.head(25), x="X", y="Y", color="character")
fig.show()

In [29]:
import pickle
with open('data.pkl', 'wb') as f:
	pickle.dump(new_df,f)
